<a href="https://colab.research.google.com/github/7REVOLUTiOn/UPSIS/blob/main/%D0%A3%D0%9F%D0%98%D0%A1_%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%E2%84%964_%D0%A0%D0%B0%D0%B7%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D1%81%D0%BF%D0%B5%D1%86%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D0%B8_%D1%82%D1%80%D0%B5%D0%B1%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B9_%D0%BA_%D0%B8%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D0%BE%D0%B9_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B5_(SRS).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Задание №4. Разработка спецификации требований к информационной системе (SRS)**

СПЕЦИФИКАЦИЯ ТРЕБОВАНИЙ К ПРОГРАММНОМУ ОБЕСПЕЧЕНИЮ
Веб-сервис монокулярной оценки глубины и генерации 3D-моделей для печати

1. ВВЕДЕНИЕ
1.1. Цель документа
Настоящий документ определяет требования к веб-сервису для монокулярной оценки глубины и генерации 3D-моделей (STL), разрабатываемому в рамках дипломного проекта. Цель спецификации — предоставить полное и детальное описание функциональных и нефункциональных требований к системе, определить критерии приемки и служить основой для разработки, тестирования и внедрения решения к сроку 04.04.2025.

1.2. Область применения
Веб-сервис предназначен для:
*   Автоматизации процесса преобразования 2D-изображений в 3D-модели с использованием моделей глубины (MiDaS, DepthAnything и др.)
*   Генерации файлов формата STL, пригодных для 3D-печати
*   Сравнительного анализа моделей оценки глубины по метрикам качества (SSIM, L1, Smoothness)
*   Предоставления инструментов постобработки mesh-моделей (фильтрация шумов, проверка на manifold)
*   Верификации качества реконструкции на наборе данных (старинные фотографии)

Система будет использоваться тремя основными группами пользователей: исследователи/студенты (пользователи), научный руководитель (эксперт) и разработчик (администратор).

1.3. Определения, акронимы и сокращения
*   Монокулярная оценка глубины (Monocular Depth Estimation) — задача восстановления карты глубины по одному изображению
*   Карта глубины (Depth Map) — одноканальное изображение, где яркость пикселя соответствует расстоянию до камеры
*   Mesh (Меш) — полигональная сетка, представляющая 3D-объект
*   STL (Stereolithography) — формат файла, описывающий поверхность геометрии 3D-модели
*   Manifold (Многообразие) — свойство 3D-модели, означающее отсутствие дыр и самопересечений, необходимое для 3D-печати
*   MiDaS, DepthAnything, NeRF, Unet — архитектуры нейронных сетей для оценки глубины
*   SSIM (Structural Similarity Index) — метрика структурного сходства
*   L1-loss — метрика средней абсолютной ошибки
*   FLOPs (Floating Point Operations) — количество операций с плавающей запятой, мера вычислительной сложности
*   Docker — платформа для контейнеризации приложений
*   S3 — объектное хранилище данных

1.4. Пользовательские роли
**Пользователь (Исследователь/Студент)**
*   **Описание:** Посетитель сервиса, осуществляющий загрузку изображений и получение 3D-моделей
*   **Количество:** Неограниченное (в рамках нагрузочного тестирования)
*   **Технический уровень:** Средний (умеет пользоваться веб-интерфейсом)
*   **Основные задачи:**
    *   Загрузка 2D-изображений
    *   Выбор модели оценки глубины
    *   Просмотр результата (карта глубины, 3D-меш)
    *   Скачивание STL-файла
    *   Просмотр метрик качества

**Эксперт (Научный руководитель)**
*   **Описание:** Специалист, проверяющий качество работы системы и соответствие требованиям 3D-печати
*   **Количество:** 1 человек
*   **Технический уровень:** Высокий (понимает специфику 3D-моделирования)
*   **Основные задачи:**
    *   Верификация метрик качества (SSIM, L1)
    *   Проверка физических дефектов на тестовых отпечатках
    *   Согласование архитектуры и требований

**Администратор (Разработчик)**
*   **Описание:** Технический специалист, отвечающий за развертывание и поддержку системы
*   **Количество:** 1 человек
*   **Технический уровень:** Высокий (владеет навыками DevOps, ML)
*   **Основные задачи:**
    *   Настройка инфраструктуры (Docker, серверы)
    *   Управление моделями и версиями алгоритмов
    *   Мониторинг производительности и логов
    *   Резервное копирование данных

2. ФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ
2.1. Аутентификация и авторизация
**ФТ-001: Базовая авторизация**
*   **Описание:** Система должна обеспечивать доступ к административной панели для разработчика
*   **Входные данные:** Логин и пароль администратора
*   **Выходные данные:** Сессия пользователя с правами администратора
*   **Бизнес-правила:**
    *   Публичная часть доступна без авторизации
    *   Административная часть защищена паролем
    *   Сессия истекает через 30 минут бездействия
*   **Приоритет:** Высокий
*   **Критерий проверки:** Успешный вход в админ-панель с корректными правами

2.2. Управление моделями глубины
**ФТ-002: Выбор модели оценки глубины**
*   **Описание:** Пользователь должен иметь возможность выбрать модель для обработки (MiDaS, DepthAnything, NeRF, Unet)
*   **Входные данные:** ID выбранной модели
*   **Выходные данные:** Конфигурация пайплайна обработки
*   **Бизнес-правила:**
    *   Доступны только предварительно загруженные и протестированные модели
    *   По умолчанию выбрана модель с лучшим балансом скорости/качества
*   **Приоритет:** Критический
*   **Критерий проверки:** Система корректно переключает модели обработки без ошибок

**ФТ-003: Сравнительный анализ моделей**
*   **Описание:** Система должна предоставлять метрики качества для выбранной модели (SSIM, L1, Smoothness, Params, FLOPs)
*   **Входные данные:** Тестовый датасет, конфигурация модели
*   **Выходные данные:** Отчет с метриками качества и вычислительной сложности
*   **Бизнес-правила:**
    *   Metrics рассчитываются на валидационной выборке
    *   Данные сохраняются в базу данных для истории сравнений
*   **Приоритет:** Высокий
*   **Критерий проверки:** Отчет содержит корректные значения метрик для всех тестируемых моделей

2.3. Обработка изображений и генерация 3D
**ФТ-004: Загрузка изображения**
*   **Описание:** Пользователь должен иметь возможность загрузить 2D-изображение для обработки
*   **Входные данные:** Файл изображения (JPG, PNG, до 10 МБ)
*   **Выходные данные:** Уникальный ID задачи, статус обработки
*   **Бизнес-правила:**
    *   Проверка формата и размера файла
    *   Автоматическое изменение размера при необходимости
*   **Приоритет:** Критический
*   **Критерий проверки:** Изображение загружается, валидируется и поступает в очередь обработки

**ФТ-005: Генерация карты глубины**
*   **Описание:** Система должна преобразовать загруженное изображение в карту глубины (numpy массив)
*   **Входные данные:** Изображение, выбранная модель
*   **Выходные данные:** Карта глубины (визуализация + данные)
*   **Бизнес-правила:**
    *   Обработка выполняется в фоновом режиме (Celery/Queue)
    *   Время обработки не должно превышать лимит (см. НФТ)
*   **Приоритет:** Критический
*   **Критерий проверки:** Карта глубины генерируется корректно, визуализируется пользователю

**ФТ-006: Генерация 3D-меша из карты глубины**
*   **Описание:** Система должна преобразовать карту глубины в полигональную сетку (mesh)
*   **Входные данные:** Карта глубины, параметры высоты/масштаба
*   **Выходные данные:** 3D-меш (объект в памяти)
*   **Бизнес-правила:**
    *   Поднятие/опускание плоскости по яркости пикселя
    *   Сохранение пропорций изображения
*   **Приоритет:** Критический
*   **Критерий проверки:** 3D-меш корректно отображается в веб-вьюере (Three.js)

**ФТ-007: Экспорт в формат STL**
*   **Описание:** Система должна позволять скачать полученную модель в формате STL
*   **Входные данные:** ID обработанной модели
*   **Выходные данные:** Файл .stl для скачивания
*   **Бизнес-правила:**
    *   Файл должен быть валидным для 3D-печати (бинарный или ASCII STL)
    *   Счетчик скачиваний обновляется
*   **Приоритет:** Критический
*   **Критерий проверки:** Скачанный файл открывается в Blender/MeshLab без ошибок

2.4. Постобработка и верификация
**ФТ-008: Фильтрация шумов и редукция полигонов**
*   **Описание:** Система должна предоставлять инструменты для улучшения качества mesh (удаление шумов, decimation)
*   **Входные данные:** 3D-меш, параметры фильтрации
*   **Выходные данные:** Оптимизированный 3D-меш
*   **Бизнес-правила:**
    *   Применяются алгоритмы сглаживания и уменьшения количества полигонов
    *   Пользователь может预览 результат перед экспортом
*   **Приоритет:** Высокий
*   **Критерий проверки:** Количество полигонов уменьшается, визуальные шумы сглаживаются

**ФТ-009: Проверка требований к 3D-печати**
*   **Описание:** Система должна проверять модель на соответствие требованиям печати (толщина стенок, manifold)
*   **Входные данные:** 3D-меш
*   **Выходные данные:** Отчет о валидности (Pass/Fail), список ошибок
*   **Бизнес-правила:**
    *   Проверка на наличие дыр (non-manifold geometry)
    *   Проверка минимальной толщины стенок
*   **Приоритет:** Высокий
*   **Критерий проверки:** Система корректно выявляет модели, непригодные для печати

2.5. Управление данными и инфраструктурой
**ФТ-010: Управление датасетом**
*   **Описание:** Администратор должен иметь возможность загружать и управлять набором данных для верификации (старинные фото)
*   **Входные данные:** Архив с изображениями и метаданными
*   **Выходные данные:** Обновленная база данных тестовых изображений
*   **Бизнес-правила:**
    *   Поддержка пакетной загрузки
    *   Привязка метаданных (источник, год)
*   **Приоритет:** Средний
*   **Критерий проверки:** Датасет успешно загружен и доступен для тестирования моделей

**ФТ-011: Интеграция компонентов системы**
*   **Описание:** Все модули (backend, frontend, ML-модели, БД) должны работать как единое целое
*   **Входные данные:** Запрос пользователя
*   **Выходные данные:** Полный цикл обработки (Upload -> Process -> Download)
*   **Бизнес-правила:**
    *   Взаимодействие через API
    *   Обработка ошибок на всех этапах
*   **Приоритет:** Критический
*   **Критерий проверки:** End-to-End тест проходит успешно без ручного вмешательства

3. НЕФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ
3.1. Требования к производительности
**НФТ-П-001: Время обработки**
Время генерации карты глубины для изображения 1024x1024 не должно превышать 10 секунд на целевом оборудовании (GPU). Время генерации STL не должно превышать 5 секунд.

**НФТ-П-002: Пропускная способность**
Система должна обрабатывать до 5 одновременных запросов на генерацию без очереди. При большей нагрузке запросы ставятся в очередь с информированием пользователя.

**НФТ-П-003: Время отклика UI**
Интерфейс пользователя должен загружаться не более 2 секунд. Интерактивные элементы (вьюер 3D) должны работать с частотой не менее 30 FPS.

3.2. Требования к надежности
**НФТ-Н-001: Доступность**
Система должна быть доступна 99% времени в период разработки и тестирования (март-апрель 2025).

**НФТ-Н-002: Целостность данных**
При сбое во время обработки задача должна быть помечена как "failed", данные не должны быть потеряны. Резервное копирование БД ежедневно.

**НФТ-Н-003: Обработка ошибок**
Пользователь должен получать понятные сообщения об ошибках (например, "Неверный формат файла", "Ошибка модели"), а не стек-трейс.

3.3. Требования к безопасности
**НФТ-Б-001: Защита от загрузки вредоносных файлов**
Система должна проверять загружаемые файлы на соответствие типу (MIME-type), запрещать исполнение скриптов.

**НФТ-Б-002: Изоляция среды**
ML-модели должны запускаться в изолированных контейнерах (Docker) для предотвращения влияния на основную систему.

3.4. Требования к удобству использования
**НФТ-Ю-001: Интуитивность**
Пользователь должен mampu загрузить фото и получить STL не более чем за 3 клика без инструкции.

**НФТ-Ю-002: Визуализация**
3D-вьюер должен позволять вращать, масштабировать и.inspectировать модель перед скачиванием.

3.5. Требования к совместимости
**НФТ-С-001: Браузерная совместимость**
Поддержка современных версий Chrome, Firefox, Safari.

**НФТ-С-002: Совместимость STL**
Генерируемые файлы должны открываться в стандартном ПО для 3D-печати (Cura, PrusaSlicer) и редактирования (Blender).

3.6. Требования к масштабируемости
**НФТ-М-001: Контейнеризация**
Система должна быть полностью развернута в Docker-контейнерах для легкого переноса и масштабирования.

4. ТРЕБОВАНИЯ К ИНТЕРФЕЙСАМ
4.1. Пользовательский интерфейс
**ТИ-ПИ-001: Главная страница**
*   Кнопка загрузки изображения (Drag & Drop)
*   Выбор модели (Dropdown)
*   Кнопка "Обработать"
*   Область предпросмотра (2D + 3D)
*   Кнопка "Скачать STL"

**ТИ-ПИ-002: Страница результатов**
*   Вьюер 3D-модели (WebGL)
*   Панель метрик качества (SSIM, L1)
*   Панель настроек постобработки (ползунки фильтрации)
*   Индикатор валидности для 3D-печати

**ТИ-ПИ-003: Административная панель**
*   Список задач обработки
*   Логи системы
*   Управление моделями (добавление/удаление весов)
*   Статистика использования

4.2. Программный интерфейс (API)
**ТИ-АПИ-001: REST API для обработки**
*   `POST /api/upload/` — загрузка изображения
*   `GET /api/status/<id>/` — статус обработки
*   `GET /api/result/<id>/` — получение результатов (depth, mesh)
*   `GET /api/download/<id>/` — скачивание STL

**ТИ-АПИ-002: Внутренний API для ML**
*   endpoints для взаимодействия backend с ML-сервисом (gRPC или HTTP)

5. ОГРАНИЧЕНИЯ И ДОПУЩЕНИЯ
5.1. Ограничения
**ОГР-001: Технологические ограничения**
Разработка ведется на стеке Python (PyTorch, OpenCV), Backend (Django/FastAPI), Frontend (React/Three.js), Docker. Выбор обусловлен наличием библиотек для ML и сроком реализации.

**ОГР-002: Временные ограничения**
Срок реализации проекта строго ограничен датой **04.04.2025**. MVP должен быть готов к этой дате. Функции, не входящие в критический путь, могут быть упрощены.

**ОГР-003: Ресурсные ограничения**
Обработка моделей глубины требует GPU. Предполагается использование предоставленных университетом ресурсов или облачных решений (Colab/Kaggle) для обучения/тестирования, сервер для демо — CPU/GPU limited.

**ОГР-004: Качество входных данных**
Качество результата напрямую зависит от качества входного изображения. Система не гарантирует идеальный результат для всех типов изображений (например, абстрактное искусство).

5.2. Допущения
**ДОП-001: Доступность датасета**
Предполагается, что набор данных из старинных фотографий будет сформирован и доступен к 04.06.2025 (для этапа тестирования в полном цикле, но для MVP к 04.04 — достаточно тестовых примеров). *Корректировка под сжатый срок:* Для защиты 04.04.2025 используется уменьшенный валидационный набор (10-20 изображений).

**ДОП-002: Стабильность библиотек**
Используемые библиотеки (MiDaS, DepthAnything) остаются совместимыми с текущей версией PyTorch в период разработки.

**ДОП-003: Инфраструктура**
Университет предоставит доступ к серверу для развертывания веб-сервиса к моменту демонстрации.

6. КРИТЕРИИ ПРИЕМКИ
6.1. Функциональная приемка
**КП-Ф-001: Реализация критических функций**
Все функции с приоритетом "Критический" (Загрузка, Генерация глубины, Генерация Mesh, Экспорт STL) реализованы и работают стабильно.

**КП-Ф-002: Сравнение моделей**
Проведено сравнение минимум 4 моделей (MiDaS, DepthAnything, NeRF, Unet) по метрикам SSIM, L1, Smoothness, Params, FLOPs. Результаты зафиксированы.

**КП-Ф-003: Валидность STL**
Сгенерированные файлы открываются в Blender и слайсерах для 3D-печати без ошибок геометрии (после применения фильтров).

**КП-Ф-004: Веб-интерфейс**
Пользователь может пройти полный путь от загрузки фото до скачивания STL через веб-интерфейс без использования командной строки.

6.2. Нефункциональная приемка
**КП-НФ-001: Производительность**
Время обработки тестового изображения не превышает установленных лимитов (10 сек на глубину).

**КП-НФ-002: Развертывание**
Система развертывается одной командой (docker-compose up) на чистой машине.

**КП-НФ-003: Качество кода**
Код покрыт комментариями, соблюдается стиль (PEP8), используется Git.

6.3. Документация
**КП-Д-001: Техническая документация**
Наличие README, описание архитектуры, инструкция по запуску.

**КП-Д-002: Отчет о тестировании**
Наличие отчета с метриками качества моделей и результатами проверки 3D-печати (фотографии тестовых отпечатков).

**КП-Д-003: Презентация**
Подготовлены материалы для демонстрации работы системы (скриншоты, видео, демо-доступ).

6.4. Тестирование
**КП-Т-001: Покрытие тестами**
Критические модули (конвертация depth->mesh, валидация STL) покрыты unit-тестами.

**КП-Т-002: Физическая верификация**
Минимум 5 моделей распечатаны на 3D-принтере, дефекты зафиксированы и минимизированы методами фильтрации.

**КП-Т-003: Соблюдение сроков**
Все этапы завершены в соответствии с календарным планом (диаграммой Ганта) с финальной датой 04.04.2025.